In [1]:
!pip install psycopg2-binary pgvector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 119.1 MB/s eta 0:00:00


In [2]:
# Install dependencies
!pip install fastapi uvicorn pyngrok pydantic psycopg2 scikit-learn Pillow

In [3]:
!pip install --no-deps facenet-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 51.2 MB/s eta 0:00:00


In [4]:
!pip install python-multipart

In [5]:
import torch
from torch import nn
from fastapi import FastAPI, File, UploadFile, HTTPException
from facenet_pytorch import MTCNN
from torchvision import models, transforms
from PIL import Image
import psycopg2
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import io
from pyngrok import ngrok
import uvicorn
import nest_asyncio
from decimal import Decimal

In [6]:
from google.colab import drive

drive.mount('/content/drive')  # Mount Google Drive

# Cuma pure_backbone_model.pth (= model.features.state_dict() dari training) yang
# benar-benar dimuat untuk embedding -- lihat penjelasan lengkap di cell inisialisasi
# model di bawah. full_classification_model.pth, backbone_model.pth, dan
# classifier_backbone_model.pth tetap tersimpan di Drive yang sama sebagai referensi,
# tapi tidak dipakai di notebook serving ini.
model_pure_backbone = "/content/drive/MyDrive/AI-Engineer/CV/face-recognition/pure_backbone_model.pth"


Mounted at /content/drive


In [7]:
# Initialize FastAPI
app = FastAPI()

# Load the MTCNN face detector
# image_size=224, margin=20 disamakan dengan resolusi training (224x224) + sedikit ruang
# di sekitar wajah, sesuai dokumentasi awal di detect_and_verify_face() di bawah.
mtcnn = MTCNN(image_size=224, margin=20, keep_all=False, post_process=True)

# Define the EfficientNetB0 architecture (backbone)
class EfficientNetB0Embedder(nn.Module):
    def __init__(self):
        super(EfficientNetB0Embedder, self).__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        self.backbone.classifier[1] = nn.Identity()  # Remove the classification layer

    def forward(self, x):
        return self.backbone(x)

# Load the trained EfficientNetB0 model for face embedding
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EfficientNetB0Embedder().to(device)

# PENTING: pure_backbone_model.pth (= model.features.state_dict() saat training) dimuat
# langsung ke submodule `backbone.features`, BUKAN ke seluruh `backbone`.
#
# Sebelumnya kode ini memuat backbone_model.pth (hasil nn.Sequential(features, avgpool))
# ke seluruh `model` pakai strict=False. Nama parameter di checkpoint itu berawalan "0."
# (indeks Sequential), sedangkan nama parameter yang dibutuhkan model ini berawalan
# "backbone.features.". Karena strict=False, mismatch ini tidak pernah error -- semua
# key checkpoint diam-diam di-skip, sehingga backbone tetap berjalan dengan bobot acak
# (weights=None, bukan hasil training). Fix ini memuat langsung ke submodule yang tepat
# supaya nama parameternya cocok, dan strict=True supaya kalau masih ada mismatch,
# errornya kelihatan jelas -- bukan gagal diam-diam lagi.
state_dict = torch.load(model_pure_backbone, map_location=device)
model.backbone.features.load_state_dict(state_dict, strict=True)
model.eval()

# Define transformation for face embedding
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [8]:
from psycopg2 import sql  # psycopg2 sendiri sudah diimport di cell import utama

# ====== Koneksi Supabase ======
# Ambil connection string dari Colab Secrets (key: SUPABASE_DB_URL), fallback ke input
# manual kalau belum diset. Didapat dari dashboard Supabase: Project Settings > Database
# > Connection string > URI. Pakai mode "Session pooler" atau "Transaction pooler"
# (bukan "Direct connection") supaya tetap konek dari jaringan Colab yang IPv4.
from google.colab import userdata
import getpass

try:
    SUPABASE_DB_URL = userdata.get('SUPABASE_DB_URL')
except Exception:
    SUPABASE_DB_URL = None

if not SUPABASE_DB_URL:
    SUPABASE_DB_URL = getpass.getpass("SUPABASE_DB_URL tidak ditemukan di Colab Secrets, masukkan manual: ")

VECTOR_SIZE = 1280  # Output size dari EfficientNetB0

# Supabase sudah menyediakan database-nya sendiri -- beda dari Postgres lokal, di sini
# tidak perlu langkah "buat database baru". Cukup pastikan extension pgvector aktif dan
# tabel face_embeddings tersedia di database itu.
try:
    conn = psycopg2.connect(SUPABASE_DB_URL)
    cur = conn.cursor()

    # Pastikan ekstensi pgvector aktif
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

    # Buat tabel face_embeddings
    cur.execute(sql.SQL("""
        CREATE TABLE IF NOT EXISTS face_embeddings (
            id SERIAL PRIMARY KEY,
            person_name TEXT NOT NULL,
            face_vector vector(%s) NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    """), [VECTOR_SIZE])

    conn.commit()
    print("Tabel `face_embeddings` siap dipakai di Supabase.")

except Exception as e:
    print("Gagal menyiapkan database Supabase:", e)

finally:
    if 'cur' in locals():
        cur.close()
    if 'conn' in locals():
        conn.close()

Tabel `face_embeddings` siap dipakai di Supabase.


In [9]:
# Connect to Supabase (Postgres + pgvector)
def get_db_connection():
    return psycopg2.connect(SUPABASE_DB_URL)

# ---------------- Normalization & Embedding ----------------
def _l2_normalize(vec: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    v = vec.astype(np.float32).reshape(-1)
    return v / (np.sqrt((v * v).sum()) + eps)

@torch.no_grad()
def get_face_embedding(face_image: Image.Image) -> np.ndarray:
    """
    Input : PIL.Image (hasil deteksi yang sudah aligned & berukuran benar)
    Output: 1D np.ndarray float32, L2-normalized
    """
    model.eval()
    x = data_transforms(face_image).unsqueeze(0).to(device)
    emb = model(x)
    if isinstance(emb, (list, tuple)):
        emb = emb[0]
    emb = emb.detach().cpu().numpy().reshape(-1).astype(np.float32)
    emb = _l2_normalize(emb)  # penting agar cosine stabil
    return emb

# ---------------- Detection (aligned; tanpa resize lagi) ----------------
def detect_and_verify_face(image_bytes: bytes) -> Image.Image:
    """
    Kembalikan wajah hasil alignment MTCNN sebagai PIL.Image.
    Syarat: tepat 1 wajah terdeteksi.
    Catatan: pastikan MTCNN diinisialisasi: MTCNN(image_size=224, margin=20, keep_all=False, post_process=True)
    """
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    # mtcnn(img) mengembalikan tensor (3, image_size, image_size) atau None bila tidak yakin
    face_tensor = mtcnn(img)  # keep_all=False → hanya satu wajah (aligned, 224x224 sesuai image_size)
    if face_tensor is None:
        raise HTTPException(status_code=400, detail="Wajah tidak terdeteksi dengan confidence memadai.")

    # Validasi hanya 1 wajah (safety guard)
    if not isinstance(face_tensor, torch.Tensor) or face_tensor.ndim != 3:
        raise HTTPException(status_code=400, detail="Terdeteksi lebih dari satu wajah. Harus tepat satu wajah.")

    # Konversi ke PIL untuk konsistensi dengan data_transforms
    face_img = transforms.ToPILImage()(face_tensor)  # sudah 224x224 dari MTCNN → TIDAK perlu .resize lagi
    return face_img

In [10]:
import ast

# ====== Konstanta & Helper ======
COSINE_THRESHOLD = 0.85  # tuning: 0.80–0.93

def _coerce_vector_to_float32(v) -> np.ndarray:
    """
    Robust converter dari berbagai format pgvector ke np.float32 (1D).
    Menangani: list float/str/Decimal, numpy, memoryview, string "[...]" atau "a,b,c".
    """
    # memoryview -> bytes -> str
    if isinstance(v, memoryview):
        v = bytes(v).decode("utf-8", errors="ignore")

    # string -> coba literal_eval; fallback split koma
    if isinstance(v, str):
        s = v.strip()
        try:
            parsed = ast.literal_eval(s)
        except Exception:
            s = s.strip("[]{}()")
            parts = [p.strip() for p in s.split(",") if p.strip()]
            parsed = [float(p) for p in parts]
        v = parsed

    if isinstance(v, np.ndarray):
        return v.astype(np.float32).reshape(-1)

    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, Decimal):
                out.append(float(item))
            elif isinstance(item, (int, float, np.floating)):
                out.append(float(item))
            elif isinstance(item, str):
                out.append(float(item))
            else:
                # coba tolist() kalau ada
                if hasattr(item, "tolist"):
                    out.extend([float(x) for x in item.tolist()])
                else:
                    raise HTTPException(status_code=500, detail=f"Tipe elemen vector tidak didukung: {type(item).__name__}")
        return np.asarray(out, dtype=np.float32).reshape(-1)

    if hasattr(v, "tolist"):
        return np.asarray(v.tolist(), dtype=np.float32).reshape(-1)

    raise HTTPException(status_code=500, detail=f"Tipe face_vector tidak dikenali: {type(v).__name__}")

# ====== FastAPI endpoints ======
@app.get("/")
def read_root():
    return {"message": "Welcome to the Face Recognition API"}

# Endpoint 1: Face Register
@app.post("/face_register/")
async def face_register(name: str, file: UploadFile = File(...)):
    name = name.strip()  # hindari mismatch di verification/recognition gara-gara spasi nyangkut
    image_bytes = await file.read()
    conn = None
    cur = None
    try:
        face = detect_and_verify_face(image_bytes)      # aligned + resize
        face_embedding = get_face_embedding(face)       # L2-normalized (pastikan fungsi sesuai)

        conn = get_db_connection()
        cur = conn.cursor()
        cur.execute(
            "INSERT INTO face_embeddings (person_name, face_vector) VALUES (%s, %s)",
            (name, face_embedding.tolist())
        )
        conn.commit()
        return {"status": "Face registered successfully.", "name": name, "dim": int(face_embedding.shape[0])}

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        if cur is not None:
            cur.close()
        if conn is not None:
            conn.close()

# Endpoint 2: Face Verification
@app.post("/face_verification/")
async def face_verification(name: str, file: UploadFile = File(...)):
    name = name.strip()
    image_bytes = await file.read()
    conn = None
    cur = None
    try:
        face = detect_and_verify_face(image_bytes)
        probe = get_face_embedding(face)                # sudah L2-normalized

        conn = get_db_connection()
        cur = conn.cursor()
        cur.execute("SELECT face_vector FROM face_embeddings WHERE person_name = %s", (name,))
        row = cur.fetchone()
        if row is None:
            raise HTTPException(status_code=404, detail="Face not found in database.")

        stored = _coerce_vector_to_float32(row[0])
        stored = _l2_normalize(stored)                  # idempoten jika sudah normalized saat simpan

        similarity = float(np.dot(probe, stored))       # cosine pada vektor L2-normalized
        return {"similarity": similarity, "verification": similarity >= COSINE_THRESHOLD}

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        if cur is not None:
            cur.close()
        if conn is not None:
            conn.close()

# Endpoint 3: Face Recognition
@app.post("/face_recognition/")
async def face_recognition(file: UploadFile = File(...)):
    image_bytes = await file.read()
    conn = None
    cur = None
    try:
        face = detect_and_verify_face(image_bytes)
        probe = get_face_embedding(face)                # L2-normalized

        conn = get_db_connection()
        cur = conn.cursor()
        cur.execute("SELECT person_name, face_vector FROM face_embeddings")
        rows = cur.fetchall()
        if not rows:
            raise HTTPException(status_code=400, detail="Database embedding kosong.")

        best_name, best_sim = None, -1.0
        for person_name, stored_vec in rows:
            v = _coerce_vector_to_float32(stored_vec)
            v = _l2_normalize(v)
            sim = float(np.dot(probe, v))               # cosine pada vektor L2-normalized
            if sim > best_sim:
                best_sim, best_name = sim, person_name

        return {"best_match": best_name, "similarity": best_sim, "recognized": best_sim >= COSINE_THRESHOLD}

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        if cur is not None:
            cur.close()
        if conn is not None:
            conn.close()

In [11]:
# https://ngrok.com/
# Ambil authtoken dari Colab Secrets (key: NGROK_AUTHTOKEN). Kalau belum diset di
# Secrets (ikon kunci di sidebar kiri Colab), fallback ke input manual (getpass,
# tidak tampil di layar dan tidak ke-print ke output cell).
from google.colab import userdata
import getpass

try:
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    NGROK_AUTHTOKEN = None

if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = getpass.getpass("NGROK_AUTHTOKEN tidak ditemukan di Colab Secrets, masukkan manual: ")

!ngrok config add-authtoken {NGROK_AUTHTOKEN}

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [12]:
import threading

# Function to run FastAPI server
def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start ngrok and FastAPI server
ngrok_tunnel = ngrok.connect(8000)
print(f"Public URL: {ngrok_tunnel.public_url}")

# Run FastAPI in a thread
thread = threading.Thread(target=run_app)
thread.start()

Public URL: https://tactile-diploma-dejected.ngrok-free.dev


In [13]:
def get_all_face_embeddings():
    """
    Mengambil semua data dari tabel face_embeddings.
    """
    try:
        conn = get_db_connection()
        cur = conn.cursor()

        # Ambil semua data dari tabel
        cur.execute("""
            SELECT id, person_name, face_vector, created_at
            FROM face_embeddings
            ORDER BY created_at DESC;
        """)

        rows = cur.fetchall()
        result = []
        for row in rows:
            result.append({
                "id": row[0],
                "person_name": row[1],
                "face_vector": row[2],  # ini array pgvector
                "created_at": row[3]
            })

        return result

    except Exception as e:
        print("❌ Gagal mengambil data:", e)
        return []

    finally:
        if 'cur' in locals():
            cur.close()
        if 'conn' in locals():
            conn.close()

In [14]:
embeddings = get_all_face_embeddings()
for item in embeddings:
    vec = _coerce_vector_to_float32(item['face_vector'])  # parsing yang benar, bukan len() string mentah
    print(f"{item['id']} - {item['person_name']} - ({vec.shape[0]} dimensi)")

INFO:     Started server process [2877]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2 - Robert Downey jr  - (1280 dimensi)


In [15]:
def delete_face_embedding_by_name(name: str):
    try:
        conn = get_db_connection()
        cur = conn.cursor()

        # Hapus baris berdasarkan person_name
        cur.execute("DELETE FROM face_embeddings WHERE person_name = %s", (name,))
        conn.commit()

        deleted = cur.rowcount
        cur.close()
        conn.close()

        if deleted == 0:
            print(f"⚠️ Tidak ada embedding untuk nama '{name}'.")
        else:
            print(f"✅ {deleted} embedding untuk '{name}' berhasil dihapus.")

    except Exception as e:
        print("❌ Gagal menghapus data embedding:", e)

In [17]:
delete_face_embedding_by_name("robert")

✅ 1 embedding untuk 'robert' berhasil dihapus.


---

**Face Recognition — EfficientNet-B0 Embeddings + pgvector**

Bagian dari eksplorasi belajar AI Engineering pribadi saya.

Blog: https://shaka-ai.hashnode.dev · GitHub: https://github.com/arielshakaramiro